In [1]:
%load_ext autoreload
%autoreload 2

In [7]:
import corc.graph_metrics.neb
import corc.utils
import corc.our_datasets
import corc.our_algorithms
import os
import pickle
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np 
import tqdm
import time
import scipy

cache_path="../../cache"

pd.set_option('display.float_format', lambda x: f'{x:.2f}')
pd.set_option('display.precision', 2)
pd.set_option('display.width', 2000)          # line width for console output


In [5]:
# hierarchical_algorithms = ["TMM-NEB"]
# hierarchical_algorithms = ["Agglomerative Clustering"]
hierarchical_algorithms = corc.our_algorithms.HIERARCHICAL_ALGORITHMS
# datasets = ["densired_circles_16"]
datasets = corc.our_datasets.CORE_HD_DATASETS

start_time = time.time()
all_purities = dict()
for algorithm in hierarchical_algorithms:
    all_purities[algorithm] = dict()
    for dataset in tqdm.tqdm(datasets):
        purities = list()
        if dataset.startswith("densired"):
            for index in range(10):
                # print(f"Processing {dataset} index {index}")
                X,y,_ = corc.utils.load_dataset(dataset, index=index, cache_path=cache_path)
                algos = corc.utils.load_algorithms(dataset, algorithm, cache_path=cache_path, index=index)
                if algos is not None:
                    purity_scores = corc.utils.get_purity_scores(algos, X, y)
                    purities.append(np.mean(purity_scores))
        else:
            X,y,_ = corc.utils.load_dataset(dataset, cache_path=cache_path)
            algos = corc.utils.load_algorithms(dataset, algorithm, cache_path=cache_path)
            if algos is not None:
                purity_scores = corc.utils.get_purity_scores(algos, X, y)
                purities.append(purity_scores)
        all_purities[algorithm][dataset] = purities

print(f"overall computation took {time.time() - start_time:.2f} seconds")

 42%|████▏     | 5/12 [00:24<00:33,  4.81s/it]/mnt/vast-nhr/projects/nim00012/git_martin/tneb_clustering/src/corc/graph_metrics/neb.py:497: RuntimeWarning: overflow encountered in add
  normed_adj += np.min(normed_adj)
100%|██████████| 12/12 [00:51<00:00,  4.26s/it]

overall computation took 199.15 seconds


In [6]:
purity_df = pd.DataFrame(
    {
        algo: {
            ds: np.mean(scores) if isinstance(scores, list) else scores
            for ds, scores in ds_dict.items()
        }
        for algo, ds_dict in all_purities.items()
    }
).T
print(purity_df)

                           densired_circles_8  densired_circles_16  densired_circles_32  densired_circles_64  densired_studt_8  densired_studt_16  densired_studt_32  densired_studt_64  mnist8  mnist16  mnist32  mnist64
Agglomerative\nClustering                0.89                 0.96                 0.93                 0.97              0.89               0.91               0.92               0.93    0.81     0.74     0.73     0.66
Single\nLinkage                          0.72                 0.85                 0.89                 0.96              0.65               0.62               0.60               0.55    0.38     0.36     0.37     0.30
Average\nLinkage                         0.91                 0.92                 0.91                 0.95              0.85               0.83               0.80               0.75    0.81     0.69     0.57     0.36
Complete\nLinkage                        0.86                 0.88                 0.88                 0.90              0.

In [6]:
purity_df.to_latex(index=False)

'\\begin{tabular}{rrrrrrrrrrrr}\n\\toprule\ndensired_circles_8 & densired_circles_16 & densired_circles_32 & densired_circles_64 & densired_studt_8 & densired_studt_16 & densired_studt_32 & densired_studt_64 & mnist8 & mnist16 & mnist32 & mnist64 \\\\\n\\midrule\n0.893041 & 0.958508 & 0.928133 & 0.970601 & 0.893594 & 0.905195 & 0.915860 & 0.925710 & 0.813451 & 0.737106 & 0.725949 & 0.656518 \\\\\n0.716534 & 0.850060 & 0.892598 & 0.964166 & 0.647608 & 0.619336 & 0.596938 & 0.550067 & 0.382833 & 0.360786 & 0.369702 & 0.300537 \\\\\n0.909341 & 0.919480 & 0.911732 & 0.946804 & 0.846672 & 0.828075 & 0.799383 & 0.749726 & 0.807423 & 0.690086 & 0.571323 & 0.359958 \\\\\n0.863462 & 0.879329 & 0.875018 & 0.904729 & 0.764201 & 0.807792 & 0.826091 & 0.840891 & 0.475887 & 0.333341 & 0.217484 & 0.192817 \\\\\n0.816568 & 0.843952 & 0.577699 & 0.582487 & 0.657877 & 0.731192 & 0.530569 & NaN & NaN & NaN & NaN & NaN \\\\\n0.984454 & 0.999727 & 0.998403 & 0.996479 & 0.914452 & 0.928518 & 0.930546 & 0.84

In [4]:
def _fmt_mean_std(val):
    if isinstance(val, list):
        mean = np.mean(val)
        std  = np.std(val, ddof=1)         
        return f"{mean:.2f}±{std:.2f}"    
    # single number → just format it (no std)
    return f"{val:.2g}"

# Build DataFrame where each cell holds “mean ± std” (or the value itself)
purity_df = pd.DataFrame(
    {
        algo: {
            ds: _fmt_mean_std(scores)
            for ds, scores in ds_dict.items()
        }
        for algo, ds_dict in all_purities.items()
    }
).T

print(purity_df)

# Convert to LaTeX – keep the “±” symbol by disabling escaping
latex_str = purity_df.to_latex(index=True, escape=False)

with open("../../figures/table_purity.tex", "wb") as f:
    f.write(latex_str.encode("utf-8"))

                          densired_circles_8 densired_circles_16 densired_circles_32 densired_circles_64 densired_studt_8 densired_studt_16 densired_studt_32 densired_studt_64     mnist8    mnist16    mnist32    mnist64
Agglomerative\nClustering          0.89±0.07           0.96±0.04           0.93±0.07           0.97±0.04        0.89±0.03         0.91±0.03         0.92±0.03         0.93±0.03  0.81±0.00  0.74±0.00  0.73±0.00  0.66±0.00
Single\nLinkage                    0.72±0.16           0.85±0.15           0.89±0.11           0.96±0.05        0.65±0.10         0.62±0.09         0.60±0.10         0.55±0.11  0.38±0.00  0.36±0.00  0.37±0.00  0.30±0.00
Average\nLinkage                   0.91±0.08           0.92±0.09           0.91±0.07           0.95±0.06        0.85±0.06         0.83±0.06         0.80±0.04         0.75±0.09  0.81±0.00  0.69±0.00  0.57±0.00  0.36±0.00
Complete\nLinkage                  0.86±0.07           0.88±0.07           0.88±0.08           0.90±0.07        0.76±0.0

/mnt/vast-nhr/home/ritzert/u14643/micromamba/envs/tneb/lib/python3.11/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/mnt/vast-nhr/home/ritzert/u14643/micromamba/envs/tneb/lib/python3.11/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [10]:
for dataset in datasets:
    # if dataset.startswith("densired"):
    #     continue
    print(f"\nDataset: {dataset}")
    # algo1 = "TMM-NEB"
    # algo1 = "Leiden"
    for algo1 in hierarchical_algorithms:
        for algo2 in hierarchical_algorithms:
            if algo1 == algo2:
                continue
            aris1 = all_purities[algo1][dataset]
            aris2 = all_purities[algo2][dataset]
            if len(aris1) < 2:
                aris1 = np.ones(10)*aris1[0]
            if len(aris2) < 2:
                aris2 = np.ones(10)*aris2[0]                
            t_stat, p_val = scipy.stats.ttest_ind(aris1, aris2)

            clean_algo1 = algo1.replace("\n"," ")
            clean_algo2 = algo2.replace("\n"," ")
            if p_val > 0.05:
                print(f"{clean_algo1} vs {clean_algo2}: t={t_stat:.2f}, p={p_val:.3f}")
# scipy.stats.ttest_ind(all_aris['TMM-NEB'][dataset], all_aris[])


Dataset: densired_circles_8
Agglomerative Clustering vs Average Linkage: t=-0.48, p=0.636
Agglomerative Clustering vs Complete Linkage: t=0.98, p=0.341
Agglomerative Clustering vs BHC: t=1.86, p=0.079
Single Linkage vs BHC: t=-1.61, p=0.124
Average Linkage vs Agglomerative Clustering: t=0.48, p=0.636
Average Linkage vs Complete Linkage: t=1.37, p=0.187
Complete Linkage vs Agglomerative Clustering: t=-0.98, p=0.341
Complete Linkage vs Average Linkage: t=-1.37, p=0.187
Complete Linkage vs BHC: t=1.15, p=0.265
BHC vs Agglomerative Clustering: t=-1.86, p=0.079
BHC vs Single Linkage: t=1.61, p=0.124
BHC vs Complete Linkage: t=-1.15, p=0.265

Dataset: densired_circles_16
Agglomerative Clustering vs Average Linkage: t=1.29, p=0.215
Single Linkage vs Average Linkage: t=-1.27, p=0.221
Single Linkage vs Complete Linkage: t=-0.56, p=0.580
Single Linkage vs BHC: t=0.12, p=0.909
Average Linkage vs Agglomerative Clustering: t=-1.29, p=0.215
Average Linkage vs Single Linkage: t=1.27, p=0.221
Average

/mnt/vast-nhr/home/ritzert/u14643/micromamba/envs/tneb/lib/python3.11/site-packages/scipy/stats/_axis_nan_policy.py:523: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
